# Bonus B: Hybrid Search

**When to use this module:** After Module 6 (Improving RAG), if the group has time.

**What you'll learn:** Why pure vector search sometimes fails, how BM25 keyword search complements it, and how to combine both using Reciprocal Rank Fusion.

**Duration:** ~40 minutes

---

## The problem with pure vector search

Vector search is good at semantic similarity: it can match 'How do I cancel my subscription?' with a chunk about 'terminating your account' even if the words don't overlap. But it has a known weakness: **exact matches on specific terms score poorly**.

Try this mentally: you have a document corpus containing product specs, and a user asks _'What is the battery capacity of the X200 Pro?'_. Vector search will retrieve chunks that are semantically similar to battery questions, but it might miss the exact chunk that mentions 'X200 Pro' if that phrase doesn't embed close to the query. A keyword search would find it immediately.

Hybrid search solves this by running **both** approaches in parallel and merging the results.

In [ ]:
# rank-bm25 is part of the bonus extra: run `uv sync --extra bonus` in a terminal,
# then restart the kernel

import chromadb
import numpy as np
from chromadb.utils import embedding_functions
from rank_bm25 import BM25Okapi

from ragsst.utils import list_files, read_file, split_text

DATA_PATH = '../data/sample_docs'
EMBEDDING_MODEL = 'multi-qa-mpnet-base-cos-v1'  # the RAGTool default from the main modules

---

## Part 1: BM25 keyword search

BM25 (Best Match 25) is a classic information retrieval algorithm from 1994
that is still one of the strongest baselines in search.
It scores documents by how often query terms appear, with two corrections:

1. **Term frequency saturation**: the score for a term grows sub-linearly with frequency.
   A document that mentions "battery" 20 times is not 20× more relevant than one that
   mentions it once.

2. **Document length normalisation**: longer documents are penalised slightly, since
   they contain more terms and would otherwise score higher by chance.

The BM25 score for a query term t in document d is:

```
IDF(t) × (tf(t,d) × (k1 + 1)) / (tf(t,d) + k1 × (1 - b + b × |d|/avgdl))
```

Where:
- `IDF(t)` = inverse document frequency (rare terms score higher)
- `tf(t,d)` = term frequency in document d
- `k1` (default 1.5) = saturation constant
- `b` (default 0.75) = length normalisation factor
- `|d|/avgdl` = document length relative to average

You do not need to remember the formula. The important points are:
- **BM25 is excellent at exact and near-exact matches**
- **BM25 has no semantic understanding** ("car" and "automobile" are completely different tokens)
- **BM25 is fast**: no neural network, only arithmetic over a term-frequency index

That covers what vector search is bad at: exact matches on
product codes, names, numbers, jargon, and rare terms.

In [ ]:
# Load and chunk documents
files = list_files(DATA_PATH, extensions=('.txt', '.pdf'))
all_chunks = []
for f in files:
    text = read_file(f)
    all_chunks.extend(split_text(text))

print(f'Total chunks: {len(all_chunks)}')
print('\nSample chunk:')
print(all_chunks[0][:200])

In [ ]:
# Build a BM25 index
# BM25 works on tokens. Splitting on whitespace alone would keep punctuation ("servers?") and
# let words like "the" or "I" decide the ranking, so we drop both.
import re

STOPWORDS = set(
    'a an and are as at be by can do does for from how i in is it my of on or the to '
    'what when where which who why with you your'.split()
)


def tokenise(text: str) -> list[str]:
    return [w for w in re.findall(r'[\w-]+', text.lower()) if w not in STOPWORDS]


tokenised_chunks = [tokenise(chunk) for chunk in all_chunks]
bm25 = BM25Okapi(tokenised_chunks)
print('BM25 index built.')

In [ ]:
def bm25_search(query: str, n: int = 5) -> list[tuple[str, float]]:
    """Return top-n chunks and their BM25 scores for a query."""
    tokenised_query = tokenise(query)
    scores = bm25.get_scores(tokenised_query)
    top_indices = np.argsort(scores)[::-1][:n]
    return [(all_chunks[i], scores[i]) for i in top_indices]


# Try it
query = 'artificial intelligence Berlin Brandenburg'
results = bm25_search(query, n=3)

print(f'BM25 results for: "{query}"\n')
for i, (chunk, score) in enumerate(results):
    print(f'Rank {i+1} (score: {score:.3f})')
    print(chunk[:200])
    print()

**To do:** Try a query with a very specific term that only appears once in the corpus (a proper noun, an acronym, a product code such as `OPS-RES` or `NL-X14` from the handbook). Compare BM25 results to what you'd expect from vector search.

---

## Part 2: Vector search (recap)

You built this in Module 2. We'll set it up quickly here so we can compare.

In [ ]:
# Build a ChromaDB collection from the same chunks
client = chromadb.Client()
embedding_func = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name=EMBEDDING_MODEL
)

collection = client.get_or_create_collection(
    name='hybrid_demo',
    embedding_function=embedding_func,
    metadata={'hnsw:space': 'cosine'},
)

collection.add(
    documents=all_chunks,
    ids=[f'chunk_{i}' for i in range(len(all_chunks))],
)
print(f'ChromaDB collection ready with {collection.count()} chunks.')

In [ ]:
def vector_search(query: str, n: int = 5) -> list[tuple[str, float]]:
    """Return top-n chunks and their cosine similarity scores."""
    results = collection.query(query_texts=[query], n_results=n)
    chunks = results['documents'][0]
    # ChromaDB returns cosine distance; convert to similarity
    similarities = [1 - d for d in results['distances'][0]]
    return list(zip(chunks, similarities))


# Try the same query
results = vector_search(query, n=3)

print(f'Vector search results for: "{query}"\n')
for i, (chunk, score) in enumerate(results):
    print(f'Rank {i+1} (similarity: {score:.3f})')
    print(chunk[:200])
    print()

---

## Part 3: Reciprocal Rank Fusion

Now we have two ranked lists. How do we combine them?

**Reciprocal Rank Fusion (RRF)** is a simple but effective method. Instead of trying to normalise scores across the two systems (which have completely different scales), it only uses the *rank* of each document in each list:

```
RRF_score(doc) = Σ 1 / (k + rank_in_list)
```

where `k` is a smoothing constant (typically 60) that reduces the impact of very high ranks. Documents that rank highly in *both* lists get a high combined score. Documents that appear in only one list get a moderate score. Documents that don't appear in either get nothing.

In [ ]:
def reciprocal_rank_fusion(
    ranked_lists: list[list[str]],
    k: int = 60
) -> list[tuple[str, float]]:
    """
    Combine multiple ranked lists using Reciprocal Rank Fusion.

    Args:
        ranked_lists: Each inner list is a ranking of document strings,
                      best first.
        k: Smoothing constant. Higher k reduces the advantage of top ranks.

    Returns:
        List of (document, rrf_score) sorted by score descending.
    """
    scores: dict[str, float] = {}

    for ranked_list in ranked_lists:
        for rank, doc in enumerate(ranked_list, start=1):
            scores[doc] = scores.get(doc, 0.0) + 1.0 / (k + rank)

    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

In [ ]:
def hybrid_search(query: str, n: int = 5) -> list[tuple[str, float]]:
    """Combine BM25 and vector search results using RRF."""

    # Get top-2n results from each method (we'll fuse down to n)
    bm25_results = bm25_search(query, n=n * 2)
    vector_results = vector_search(query, n=n * 2)

    bm25_ranking = [chunk for chunk, _ in bm25_results]
    vector_ranking = [chunk for chunk, _ in vector_results]

    fused = reciprocal_rank_fusion([bm25_ranking, vector_ranking])
    return fused[:n]

In [ ]:
# Compare all three approaches on the same query
query = 'artificial intelligence Berlin Brandenburg'
n = 3

bm25_top = [c for c, _ in bm25_search(query, n)]
vector_top = [c for c, _ in vector_search(query, n)]
hybrid_top = [c for c, _ in hybrid_search(query, n)]

print('=' * 70)
print(f'Query: "{query}"')
print('=' * 70)

for label, results in [('BM25', bm25_top), ('Vector', vector_top), ('Hybrid', hybrid_top)]:
    print(f'\n--- {label} top-{n} ---')
    for i, chunk in enumerate(results):
        in_all = chunk in bm25_top and chunk in vector_top and chunk in hybrid_top
        overlap = '✓ in all 3' if in_all else ''
        print(f'  {i+1}. {chunk[:100].strip()}... {overlap}')

**To do:**
1. Try a semantic query (e.g. 'places to eat in Europe'). Which method performs best?
2. Try an exact-match query (a proper noun or acronym from the documents). Which method performs best?
3. Experiment with the `k` parameter in RRF. What happens at `k=1`? At `k=100`?

Neither BM25 nor vector search wins on every type of query. Hybrid search hedges between them: it rarely fails completely, but on a given set of questions it can still lose to the better of the two. The next cell measures it.

### Measure it

Trying single queries shows *how* the methods differ. Whether hybrid search is *better* for your
documents is a question for the gold set from Module 5.

In [ ]:
from ragsst.evaluation import load_gold_dataset, recall_at_k

gold_dataset = load_gold_dataset('../data/gold_dataset.json')

for name, search in [('BM25', bm25_search), ('vector', vector_search), ('hybrid', hybrid_search)]:
    def retrieve(query: str, k: int, search=search) -> list[str]:
        return [chunk for chunk, _ in search(query, k)]

    r1 = recall_at_k(retrieve, gold_dataset, k=1)
    r3 = recall_at_k(retrieve, gold_dataset, k=3)
    print(f'{name:<8} Recall@1={r1:.2f}  Recall@3={r3:.2f}')

In our runs hybrid search did *not* beat plain vector search on this gold set: most questions are
written in everyday words that do not appear in the handbook, which is where BM25 is weakest.
Hybrid search pays off when users type the exact terms of the documents: product codes, names,
error messages. Which kind of questions do *your* users ask?

---

## Part 4: Plug into the RAG pipeline

Hybrid search is a drop-in replacement for the retrieval step. Everything else stays the same.

In [ ]:
import json

import requests

from ragsst.parameters import LLMBASEURL

MODEL = 'llama3.2:3b'


def get_context_prompt(query: str, context: str) -> str:
    # Same template as RAGTool.get_context_prompt in Module 4
    return (
        'Use only the following context to answer the query at the end. '
        'If the answer is not in the context, say "I don\'t have that information." '
        'Do not make up answers. Keep the answer as concise as possible.\n'
        f'Context:\n{context}\nQuery:\n{query}'
    )


def generate(prompt: str) -> str:
    url = LLMBASEURL + '/generate'
    data = {'prompt': prompt, 'model': MODEL, 'stream': False}
    r = requests.post(url, json=data)
    return json.loads(r.text).get('response', '')


def hybrid_rag(query: str, n: int = 3) -> str:
    """Full RAG pipeline using hybrid retrieval."""
    retrieved = hybrid_search(query, n=n)
    context = '\n\n'.join(chunk for chunk, _ in retrieved)
    prompt = get_context_prompt(query, context)
    return generate(prompt)


# Exact product codes: the kind of query BM25 is good at
query = 'What is the difference between the NL-X14 and the NL-W16?'
print(f'Query: {query}\n')
print('Answer:')
print(hybrid_rag(query))

---

## Further reading

- [BM25 explained](https://www.elastic.co/blog/practical-bm25-part-2-the-bm25-algorithm-and-its-variables)
- [Reciprocal Rank Fusion paper](https://dl.acm.org/doi/10.1145/1571941.1572114)
- [Hybrid search in practice (Weaviate)](https://weaviate.io/blog/hybrid-search-explained)